In [0]:
%run ./01_config

In [0]:
"""
07_quality_and_exports.py  —  Data contract tests and exports

Part of the SAP PM cost-aware maintenance optimisation pipeline.
Run order is filename order: 01 through 12.
"""

# 07 — Data contract tests and exports

# Executes the Chapter 4 verification tests as code and writes the results to
# dq_results. Chapter 4 can then cite a table produced by the pipeline rather than
# asserting compliance in prose — which is the difference between claiming NFR2/NFR3
# and evidencing them.

# Test  -  Requirement  -  What it checks
# T1  -  FR1  -  Pipeline runs from the contract entities alone; all present and non-empty
# T7  -  NFR2  -  ISO 14224 class grouping and EN 13306 order-type mapping are complete
# T8  -  NFR3  -  No personal-data fields in any silver table
# T-DQ  -  —  -  Contract field coverage, censoring sanity, cost-ratio calibration

# Shared configuration from '01_config' is assumed to be in scope.

from pyspark.sql import functions as F
import datetime, re
import pandas as pd

use_project_schema()
results = []

def record(test_id, requirement, description, passed, detail):
    results.append({
        "test_id": test_id,
        "requirement": requirement,
        "description": description,
        "status": "PASS" if passed else "FAIL",
        "detail": detail,
        "dataset_version": DATASET_VERSION,
        "checked_at": datetime.datetime.now(),
    })
    print(f"[{'PASS' if passed else 'FAIL'}] {test_id} {description}: {detail}")

# T1 — pipeline executes from the contract entities alone

counts = {}
for e in CONTRACT_ENTITIES:
    name = tbl(f"bronze_{e.lower()}")
    counts[e] = spark.table(name).count() if spark.catalog.tableExists(name) else 0

empty = [e for e, n in counts.items() if n == 0]
record("T1", "FR1", "All contract entities present and non-empty",
       not empty,
       f"{'; '.join(f'{k}={v:,}' for k, v in counts.items())}"
       + (f" | EMPTY: {empty}" if empty else ""))

gold_ok = all(spark.catalog.tableExists(tbl(t)) for t in
              ["gold_survival_intervals", "gold_class_cost_params", "gold_incumbent_cycles"])
record("T1b", "FR1", "Gold decision inputs materialised from those entities only",
       gold_ok, "gold_survival_intervals / gold_class_cost_params / gold_incumbent_cycles")

# T7 — standards alignment (NFR2)

eq = spark.table(tbl("silver_equipment"))
bad_group = eq.filter(~F.col("iso14224_group").isin(ISO14224_CLASS_PREFIXES)).count()
groups = [r[0] for r in eq.select("iso14224_group").distinct().collect()]
record("T7a", "NFR2", "Every equipment class maps to an ISO 14224 group",
       bad_group == 0, f"{bad_group} unmapped; groups present: {sorted(groups)}")

orders = spark.table(tbl("silver_order"))
unmapped = orders.filter(F.col("en13306_category").isNull()).count()
mapping = (orders.groupBy("order_type", "en13306_category").count()
                 .orderBy("order_type").collect())
record("T7b", "NFR2", "Every SAP order type maps to an EN 13306 category",
       unmapped == 0,
       "; ".join(f"{r['order_type']}->{r['en13306_category']} ({r['count']:,})" for r in mapping)
       + (f" | UNMAPPED: {unmapped:,}" if unmapped else ""))

# T8 — no personal data (NFR3)

# Two parts. First, a denylist scan of column names across every silver table. Second, a
# value-level check on technician_token: the generator emits TECH-nn pseudonyms, and
# the test asserts that pattern holds rather than assuming it. Worth being explicit about
# this in Chapter 4 — a reviewer looking at AFRU will see a personnel field and want to
# know it is a synthetic token, not a redacted name.

DENYLIST = ["name", "surname", "firstname", "lastname", "email", "phone", "address",
            "birth", "dob", "ssn", "nino", "passport", "employee_id", "personnel_number",
            "user_id", "badge"]

silver_tables = [r.tableName for r in spark.sql(f"SHOW TABLES IN {CATALOG}.{SCHEMA}").collect()
                 if r.tableName.startswith("silver_")]

hits = []
for st in silver_tables:
    for c in spark.table(tbl(st)).columns:
        if any(d in c.lower() for d in DENYLIST):
            hits.append(f"{st}.{c}")

record("T8a", "NFR3", "No personal-data column names in silver layer",
       not hits, f"scanned {len(silver_tables)} tables; hits: {hits or 'none'}")

tokens = (spark.table(tbl("silver_confirmation"))
               .select("technician_token").distinct().limit(200).collect())
token_vals = [r[0] for r in tokens if r[0] is not None]
pattern = re.compile(r"^TECH-\d+$")
non_conforming = [v for v in token_vals if not pattern.match(v)]
record("T8b", "NFR3", "Personnel field contains synthetic tokens only",
       not non_conforming,
       f"{len(token_vals)} distinct values sampled, pattern TECH-<n>; "
       f"non-conforming: {non_conforming[:5] or 'none'}")

# Contract field coverage

REQUIRED = {
    "silver_notification": ["notification_id", "equipment_id", "notification_type",
                            "malfunction_start", "priority", "originating_notification_id"],
    "silver_order": ["order_id", "notification_id", "equipment_id", "order_type",
                     "basic_start", "basic_finish", "total_actual_cost"],
    "silver_confirmation": ["confirmation_id", "order_id", "actual_work_hours"],
    "silver_maintenance_plan": ["plan_id", "equipment_id", "cycle_length", "cycle_unit"],
    "silver_equipment": ["equipment_id", "equipment_class", "start_up_date",
                         "functional_location_id"],
    "silver_functional_location": ["functional_location_id", "criticality"],
}

for table, cols in REQUIRED.items():
    df = spark.table(tbl(table))
    total = df.count()
    populated = {}
    for c in cols:
        if c not in df.columns:
            populated[c] = "ABSENT"
        else:
            nn = df.filter(F.col(c).isNotNull()).count()
            populated[c] = f"{nn/total:.0%}" if total else "0%"
    problems = [c for c, v in populated.items() if v in ("ABSENT", "0%")]
    record(f"DC-{table}", "FR1",
           f"Data contract fields populated in {table}",
           not problems,
           "; ".join(f"{c}={v}" for c, v in populated.items()))

# Survival dataset sanity

iv = spark.table(tbl("gold_survival_intervals"))
neg = iv.filter(F.col("duration_days") <= 0).count()
record("SV1", "FR2", "No non-positive interval durations", neg == 0, f"{neg} bad rows")

cens = iv.select(F.round(1 - F.avg("event_observed"), 3)).collect()[0][0]
record("SV2", "FR2", "Censoring rate within a plausible band (0.20-0.90)",
       0.20 <= cens <= 0.90, f"overall censoring rate = {cens}")

split_counts = {r["split"]: r["n"] for r in
                iv.groupBy("split").agg(F.count("*").alias("n")).collect()}
test_events = iv.filter((F.col("split") == "test") & (F.col("event_observed") == 1)).count()
record("SV3", "FR5", "Temporal hold-out contains observed failures",
       test_events > 0,
       f"train={split_counts.get('train', 0):,} test={split_counts.get('test', 0):,}; "
       f"test failures={test_events:,}")

thin = (iv.groupBy("equipment_class").agg(F.sum("event_observed").alias("f"))
          .filter(F.col("f") < 30).collect())
record("SV4", "FR2", "Every class has >=30 observed failures for per-class fitting",
       not thin,
       "thin classes: " + (", ".join(f"{r['equipment_class']}({r['f']})" for r in thin) or "none"))

# Cost calibration

# Section 4.1.1 states a fleet-level breakdown-to-preventive ratio of 4.0x. This checks the
# realised ratio in the data rather than the configured one — they diverge whenever the
# per-order random multiplier and the class mix interact.

ratio_row = spark.sql(f"""
SELECT ROUND(PERCENTILE_APPROX(CASE WHEN order_class='CORRECTIVE' THEN total_actual_cost END, 0.5)
           / PERCENTILE_APPROX(CASE WHEN order_class='PREVENTIVE' THEN total_actual_cost END, 0.5), 3) AS ratio
FROM {tbl('silver_order')}
""").collect()[0]
ratio = ratio_row["ratio"]
record("CC1", "FR3", "Fleet breakdown:preventive cost ratio near the 4.0x calibration target",
       ratio is not None and 3.5 <= ratio <= 4.5, f"realised median ratio = {ratio}")

dt = spark.sql(f"""
SELECT ROUND(SUM(downtime_valuation), 2) AS total_downtime,
       ROUND(SUM(downtime_valuation) / NULLIF(SUM(total_actual_cost), 0), 3) AS downtime_share
FROM {tbl('silver_order')}
""").collect()[0]
record("CC2", "FR3", "Downtime valuation recorded separately from settled order cost",
       dt["total_downtime"] is not None and dt["total_downtime"] > 0,
       f"total downtime valuation = {dt['total_downtime']}, "
       f"{dt['downtime_share']}x settled cost")

# Covariate availability

# Notebook 06 ranks equipment using manufacturer, plant and criticality. If any of these is
# absent or constant, the prediction layer loses the discrimination signal and concordance
# collapses toward 0.5 — so it is checked here rather than discovered three notebooks later.

eqs = spark.table(tbl("silver_equipment"))
for col, table, label in [("manufacturer", eqs, "manufacturer (EQUI.HERST)"),
                          ("planning_plant", eqs, "plant (EQUI.IWERK)")]:
    total = table.count()
    nn = table.filter(F.col(col).isNotNull()).count()
    levels = table.select(col).distinct().count()
    record(f"CV-{col}", "FR2", f"{label} populated and varying",
           nn == total and levels > 1,
           f"{nn}/{total} populated, {levels} distinct levels")

fl = spark.table(tbl("silver_functional_location"))
crit_levels = fl.filter(F.col("criticality").isNotNull()).select("criticality").distinct().count()
record("CV-criticality", "FR2", "Criticality populated and varying",
       crit_levels > 1, f"{crit_levels} distinct levels")

# Persist results

dq = spark.createDataFrame(pd.DataFrame(results))
(dq.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(tbl("dq_results")))

display(spark.table(tbl("dq_results")).orderBy("test_id"))

failed = [r["test_id"] for r in results if r["status"] == "FAIL"]
print(f"\n{len(results) - len(failed)}/{len(results)} checks passed."
      + (f" FAILED: {failed}" if failed else ""))

# Exports for Appendix E

# Small enough to collect to the driver. Written as single clean CSVs to the volume so
# they can be pasted into the report as tables without post-processing.

exports = {
    "dataset_statistics": spark.sql(f"""
        SELECT 'Equipment classes' AS property, CAST(COUNT(DISTINCT equipment_class) AS STRING) AS value FROM {tbl('silver_equipment')}
        UNION ALL SELECT 'Equipment items', CAST(COUNT(*) AS STRING) FROM {tbl('silver_equipment')}
        UNION ALL SELECT 'Functional locations', CAST(COUNT(*) AS STRING) FROM {tbl('silver_functional_location')}
        UNION ALL SELECT 'Maintenance plans', CAST(COUNT(*) AS STRING) FROM {tbl('silver_maintenance_plan')}
        UNION ALL SELECT 'Notifications (QMEL)', CAST(COUNT(*) AS STRING) FROM {tbl('silver_notification')}
        UNION ALL SELECT 'Work orders (AUFK)', CAST(COUNT(*) AS STRING) FROM {tbl('silver_order')}
        UNION ALL SELECT 'Confirmations (AFRU)', CAST(COUNT(*) AS STRING) FROM {tbl('silver_confirmation')}
        UNION ALL SELECT 'Survival intervals', CAST(COUNT(*) AS STRING) FROM {tbl('gold_survival_intervals')}
        UNION ALL SELECT 'Observed failures', CAST(SUM(event_observed) AS STRING) FROM {tbl('gold_survival_intervals')}
    """),
    "class_summary": spark.sql(f"""
        SELECT equipment_class,
               COUNT(DISTINCT equipment_id) AS equipment_items,
               COUNT(*) AS intervals,
               SUM(event_observed) AS observed_failures,
               ROUND(1 - AVG(event_observed), 3) AS censoring_rate,
               ROUND(AVG(duration_days), 1) AS mean_duration_days
        FROM {tbl('gold_survival_intervals')} GROUP BY equipment_class ORDER BY equipment_class
    """),
    "class_cost_params": spark.table(tbl("gold_class_cost_params")),
    "incumbent_cycles": spark.table(tbl("gold_incumbent_cycles")),
    "dq_results": spark.table(tbl("dq_results")),
}

for name, df in exports.items():
    path = f"{EXPORTS}/{name}.csv"
    df.toPandas().to_csv(path, index=False)
    print(f"wrote {path}")

display(dbutils.fs.ls(EXPORTS))